# 02 — Modelling
Runs the exact training pipeline from `src/models/train.py` and inspects the results.

- **Task A**: cross-sectional log-price, GroupKFold by set.
- **Task B**: 30/90-day forward log return, purged expanding time splits (needs several weeks of snapshots).

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src.config import load_config, resolve_path, set_seeds

cfg = load_config()
set_seeds(cfg["seed"])
df_raw = pd.read_parquet(resolve_path(cfg, "dataset"))
df_raw.shape

In [ ]:
# Task A — full CV + Optuna search (slow: ~50 trials x 5 folds).
# For a quicker notebook run, lower the trial count here.
cfg["model"]["optuna_trials"] = 15

from src.models.train import train_task_a
metrics_a = train_task_a(cfg, df_raw)
print(json.dumps(metrics_a, indent=2, default=str))

In [ ]:
# Task B — will raise/skip unless you have >= min_train_days of snapshots
from src.models.train import train_task_b
try:
    metrics_b = train_task_b(cfg, df_raw, 30)
    print(json.dumps(metrics_b, indent=2, default=str))
except RuntimeError as exc:
    print("Task B skipped:", exc)

In [ ]:
# Side-by-side metric table from reports/metrics.json
metrics = json.loads((resolve_path(cfg, "reports_dir") / "metrics.json").read_text())
rows = []
for task, res in metrics.items():
    for model, m in res.get("metrics", {}).items():
        rows.append({"task": task, "model": model,
                     **{k: v for k, v in m.items() if not isinstance(v, dict)}})
pd.DataFrame(rows)

In [ ]:
# Out-of-fold predicted vs actual for Task A
import matplotlib.pyplot as plt
oof = pd.read_parquet(resolve_path(cfg, "reports_dir") / "oof_task_a.parquet")
plt.figure(figsize=(6, 6))
plt.scatter(oof["log_price"], oof["pred_xgb"], s=4, alpha=0.3)
lims = [oof["log_price"].min(), oof["log_price"].max()]
plt.plot(lims, lims, c="red")
plt.xlabel("actual log price"); plt.ylabel("OOF predicted")
plt.show()

In [ ]:
# SHAP + residual figures land in reports/figures/
from src.models.evaluate import _load, shap_plots, residual_plots
bundle, oof = _load(cfg, "task_a")
shap_plots(cfg, bundle, oof, "task_a")
residual_plots(cfg, bundle, oof, "task_a")
print("figures written to", resolve_path(cfg, "figures_dir"))